# Lombardy heat risk (2020–2022) — Earth Engine + geemap

Same workflow as `gee/heat_risk_lombardy_2020_2022.js`, implemented in Python (`src/lombardy_heat_risk_gee.py`).

**One-time setup (terminal):**
```bash
pip install -r requirements.txt
earthengine authenticate
```

Then open this notebook from the **repository root** (or adjust `sys.path` in the next cell).

If you use a Google Cloud Earth Engine project, use `ee.Initialize(project='your-project-id')` instead of plain `ee.Initialize()`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import ee
import geemap

from src.lombardy_heat_risk_gee import build_stack, export_image_to_drive

ee.Initialize()

In [ ]:
stack = build_stack()
aoi = stack["aoi"]

d2020, d2021, d2022 = stack["d2020"], stack["d2021"], stack["d2022"]
hazard_mean = stack["hazard_mean"]
hazard_norm = stack["hazard_norm"]
pop_density = stack["pop_density"]
risk = stack["risk"]

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 8)

Map.addLayer(aoi, {"color": "white"}, "Lombardy AOI", False)
Map.addLayer(
    d2020,
    {"min": -5, "max": 15, "palette": ["313695", "4575b4", "74add1", "fdae61", "d73027"]},
    "dT 2020",
    False,
)
Map.addLayer(
    hazard_mean,
    {"min": -5, "max": 15, "palette": ["313695", "4575b4", "74add1", "fdae61", "d73027"]},
    "dT mean 2020-22",
    True,
)
Map.addLayer(
    hazard_norm,
    {"min": 0, "max": 1, "palette": ["ffffcc", "feb24c", "fd8d3c", "f03b20", "bd0026"]},
    "H_norm",
    False,
)
Map.addLayer(
    pop_density,
    {"min": 0, "max": 8000, "palette": ["ffffd4", "fed98e", "fe9929", "d95f0e", "993404"]},
    "Pop density (/km2)",
    False,
)
Map.addLayer(
    risk,
    {"min": 0, "max": 5000, "palette": ["ffffcc", "feb24c", "fd8d3c", "f03b20", "bd0026"]},
    "Risk (H_norm x density)",
    False,
)
Map

## Export to Google Drive

Running the next cell **starts** export tasks. Monitor progress in the [Earth Engine Tasks](https://code.earthengine.google.com/tasks) page, or use `geemap.ee_export_image_to_drive` monitoring helpers.

Exports use **100 m**, **EPSG:32632**, folder **`GEE_Lombardy_HeatRisk`** (same defaults as the JS script).

In [ ]:
DRIVE_FOLDER = "GEE_Lombardy_HeatRisk"
SCALE_M = 100
CRS_EXPORT = "EPSG:32632"

exports = [
    (hazard_mean, "Lombardy_deltaT_mean_2020_2022", "lombardy_deltaT_mean_2020_2022"),
    (hazard_norm, "Lombardy_H_norm_2020_2022", "lombardy_H_norm_2020_2022"),
    (pop_density, "Lombardy_pop_density_2020_WorldPop", "lombardy_pop_density_2020"),
    (risk, "Lombardy_heat_risk_2020_2022", "lombardy_heat_risk_2020_2022"),
]

tasks = []
for img, desc, prefix in exports:
    t = export_image_to_drive(
        img,
        description=desc,
        file_name_prefix=prefix,
        region=aoi,
        folder=DRIVE_FOLDER,
        scale_m=SCALE_M,
        crs=CRS_EXPORT,
    )
    t.start()
    tasks.append(t)

print(f"Started {len(tasks)} export tasks to Drive folder '{DRIVE_FOLDER}'.")